# 🧬 Connect AI — 장기 기억 학습 (Unsloth)
내 1인 기업 지식을 모델 **가중치에 체득**시킵니다. 위 메뉴 **런타임 → 모두 실행**만 누르면 됩니다 (무료 T4 GPU).
- 데이터셋: `` (단기 지식 → conversations Q&A)
- 베이스 모델: `unsloth/Qwen2.5-3B-Instruct`  ← *내가 쓰는 모델로 바꿔도 됩니다 (누적 학습)*
- 결과 모델: `mybrain-lee` (GGUF — Connect AI 내장 엔진에 바로 로드, LM Studio 불필요)
- 설정: rank 16/alpha 32 · dropout 0 · lr 0.0002 · steps 52 · seq 1024 · linear · 양자화 q4_k_m (데이터 26개)


In [ ]:
%%capture
# 버전을 직접 고정하지 않는다 — Unsloth가 현재 Colab torch에 맞는 의존성(torchao·transformers 등)을 알아서 설치.
# (고정 레시피는 Colab torch가 바뀌면 register_constant/recompile_limit 같은 충돌이 연쇄로 난다)
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo


## 🔑 HuggingFace 로그인 (맨 먼저!)
아래 칸에 **write 토큰**을 붙여넣으세요. *비공개 데이터셋을 불러오고*, 학습된 모델을 *업로드*하는 데 둘 다 필요해요.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
# 🔐 로그인을 맨 앞에서 확인 — 안 돼 있으면 긴 학습 전에 바로 멈춰서 시간 낭비 방지
from huggingface_hub import HfApi
try:
    print("✅ 로그인됨:", HfApi().whoami()["name"], "— 결과는 내 계정에 올라가요")
except Exception:
    raise SystemExit("❌ 먼저 위 🔑 칸에 HuggingFace write 토큰을 붙여넣고 Login을 누르세요. 그다음 [런타임 → 모두 실행]을 다시 누르면 됩니다.")


In [ ]:
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    dtype = None, max_seq_length = 1024,
    load_in_4bit = True, full_finetuning = False,
)
print("✅ 베이스 모델 로딩 완료")


In [ ]:
# LoRA — 전체의 1% 미만만 학습(메모리↓, 페르소나·핵심지식엔 충분)
model = FastModel.get_peft_model(
    model, finetune_language_layers=True, finetune_attention_modules=True,
    finetune_mlp_modules=True, finetune_vision_layers=False,
    r = 16, lora_alpha = 32, lora_dropout = 0, bias = "none", random_state = 3407,
)


## 📦 단기 지식 데이터셋 (conversations Q&A)
내 지식이 **이 노트북에 직접 포함**돼 있어요 (업로드 불필요). 각 행 = `{conversations:[{user},{assistant}]}`


In [ ]:
import base64
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
# 내 지식(노트북에 포함) — base64로 안전하게 심어둠
_B64 = "eyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiLrr7jqta3sl5Ag64yA7ZW0IOyVjOugpOykmC4ifSx7InJvbGUiOiJhc3Npc3RhbnQiLCJjb250ZW50Ijoi66+46rWtIO2VmeyCrMK37ISd7IKsIO2VmeychOulvCAxMOuFhMK37YGwIOu5hOyaqeydhCDrk6Tsl6wg65WE7KeA66eMLCDsp4DquIjsnYAg6re4IOyhuOyXheyepeunjOycvOuhnCDst6jsl4XsnbQg7Ja066C164ukLiDsnbjthLQg7JqU6rG07KGw7LCoIFwi67aE7JW8IOyDgeychCAxJcK37IiY7ZWZIOyYrOumvO2UvOyVhOuTnCDsnoXsg4FcIiDsiJjspIDsnLzroZwg67mE7ZiE7Iuk7KCB7Jy866GcIOuGkuyVhOyhjOuLpC4ifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50Ijoi7KeA6riI7J2YIOq0gOugqO2VtOyEnCDshKTrqoXtlbTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IuyngOq4iOydmCDrgpjrpbwg66eM65OgIOqxtCDsobjsl4XsnqXsnbQg7JWE64uI6528IO2Vmeq1kCDri6Tri4jrqbAg7ZWcIFwi7IKs7J2065OcIO2UhOuhnOygne2KuFwi64ukLiAyMDE164WE67aA7YSwIEdpdEh1YuyXkCA4MDDqsJwg7J207IOBIO2RuOyLnCjshJzruYTsiqTtmZQg7Jew7Iq1KSwg7J247Iqk7YOAwrfsnKDtipzruIzsl5AgMTAwMOqwnCDrhJjripQg7IiP7Y+8wrfrobHtj7zsnYQg66eM65Ok66mwIOyYqOudvOyduCDruYTspojri4jsiqTsmYAg66eI7LyA7YyF7J2EIOyKpOyKpOuhnCDsnbXtmJTri6QuIn1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6Iuq3uOydtCjqsIApIOutlOyngCDslYzroKTspITrnpg/In0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6Iuq3uCDsgqzsnbTrk5wg7ZSE66Gc7KCd7Yq4IOqyve2XmOycvOuhnCDsiqTtg4Dtirjsl4XsnYQg66eM65Ok7JeI6rOgLCDsoITqs7XsnbQgQUnsmIDquLDsl5AgQUkg7JeQ7J207KCE7Yq466W8IOunjOuTpOyWtCDtmITsnqwgMeyduCDquLDsl4XsnYQg7Jq07JiB7ZWc64ukLiJ9XX0KeyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiJDT05ORUNU7JeQIOuMgO2VtCDrhKTqsIAg7JWE64qUIOqxuCDrp5DtlbTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IkNPTk5FQ1QgQUkgTEFCIOycoO2KnOu4jOuKlCDslb0gMTDrp4wg6rWs64+FLiDsoJzrjIDroZwg7ZWcIDZ+N+qwnOyblOqwhCBcIkFJIOyImOydte2ZlMK3QUkg7Iuc64yAIOyDneyhtFwi7J2EIOyjvOygnOuhnCDrobHtj7wgODDqsJzrpbwg66eM65Ok7JeI64ukLiJ9XX0KeyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiJBSeydhCjrpbwpIOygleumrO2VtOyEnCDshKTrqoXtlbTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IkFJIOyLnOuMgOyXkCDsgqzrnbzsp4QgM+qwgOyngCgzQSk6IOKRoCBBZ2Uo67Cw7JuA7J2YIOuCmOydtCkg4pGhIEFjYWRlbXko7ZWZ7JeFIOy7pOumrO2BmOufvCkg4pGiIEFuc3dlcijsoJXri7UpLiJ9XX0KeyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiLikaDsl5Ag64yA7ZW0IOyekOyEuO2eiCDslYzroKTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IuKRoCBBZ2Ug4oCUIOuwsOybgOyXkCDrgpjsnbTqsIAg7JeG7Ja07KGM64ukLiDstIjspJHqs6DCt+uMgO2VmSDsg4HqtIDsl4bsnbQg7Y+J7IOdIOqzteu2gO2VtOyVvCDtlZzri6QuIOyngOq4iOydgCDquInqsqntlZwg67OA7ZmU6riw6528IOqzoDMg7IiY64ql7IOd7LKY65+8IOqzteu2gO2VtOyVvCDtlZjripQg67mE7IOBIOyLnOq4sOuLpC4ifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50Ijoi4pGh7JeQIOuMgO2VtCDslYzroKTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IuKRoSBBY2FkZW15IOKAlCDquLDsobQg7ZWZ6rWQIOq1kOycoSDsu6TrpqztgZjrn7zsnbQg66y064SI7KGM64ukLiDtkZzrqbTtmZTrkJjsp4Ag7JWK7J2EIOu/kCwg7IiY64qlwrfrgrTsi6Ag7Iuc7Iqk7YWc64+EIOqysOq1rSDrsJTrgJDri6QuIOyDne2DnOqzhOqwgCDqsbDrjIDtlbQg6rO166Gg7ZmU6rCAIOyViCDrkKAg67+Q7J2064ukLiJ9XX0KeyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiLikaIg6rSA66Co7ZW07IScIOyEpOuqhe2VtOykmC4ifSx7InJvbGUiOiJhc3Npc3RhbnQiLCJjb250ZW50Ijoi4pGiIEFuc3dlciDigJQg7KCV64u17J20IOyXhuuKlCDshLjsg4HsnbTri6QuIOqwmeydgCDsg4Htmansl5Ag6rCZ7J2AIOyGlOujqOyFmOydhCDrgrTrj4Qg65CgIOuVjOuPhCDslYgg65CgIOuVjOuPhCDsnojri6QuIOyEuOyDgeydgCDtmZXrpaAocHJvYmFiaWxpdHkp7J2066mwIOunpOyasCDrs7XsnqEoY29tcGxleCntlZjri6QuIn1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6Iuq4sOyhtOydtCjqsIApIOutlOyngCDslYzroKTspITrnpg/In0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6Iuq4sOyhtCDqtZDsnKHsnZgg66qp7KCB7J2AIFwi7KKL7J2AIO2ajOyCrCjsgrzshLHCt0xHIOuTsSDtj4nsg53sp4HsnqUpIOy3qOyXhVwi7J207JeI64ukLiDqt7jrnpjshJwg6rWQ7JyhIOyekOyytOqwgCDtmozsgqzsl5DshJwg7IOd7KG07ZWY6riwIOychO2VnCDqtazsobDsmIDri6QuIn1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6Iuy3qOyXheyXkCDrjIDtlbQg64Sk6rCAIOyVhOuKlCDqsbgg66eQ7ZW07KSYLiJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiLst6jsl4Ug6rK97J+B7J20IOyLrO2VtOyniOyImOuhnSDquLDspIDsuZgo7ZWZ7IKs4oaS7ISd7IKs4oaS67CV7IKs4oaS7ZW07Jm4IOycoO2VmSnqsIAg6rOE7IaNIOuGkuyVhOyhjOuLpC4g7ZqM7IKsIOuTpOyWtOqwgOq4sOqwgCDsoJDsoJAg7Ja066Ck7JuM7KGM64uk64qUIOucu+ydtOqzoCwgQUnqsIAg64KY7Jik66mwIOydtCDqt5zsuZnsnbQg66y064SI7KGM64ukLiJ9XX0KeyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiLtmozsgqzsnYQo66W8KSDsoJXrpqztlbTshJwg7ISk66qF7ZW07KSYLiJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiLtmozsgqwg7LGE7Jqp7J20IOykhOqzoCDsmKTtnojroKQg64K067aAIOyduOugpeydtCDrsJbsnLzroZwg64KY7Jik6rOgIOyeiOuLpCjtgbAg6riw7JeF7J287IiY66GdIOyLrO2VqCkuIDEw66qFIOykkSAx66qF66eMIOy3qOyXhe2VmOuptCDrgpjrqLjsp4AgOeuqheydgCDqsrDqta0g7IKs7JeF7J2EIO2VoCDsiJjrsJbsl5Ag7JeG64ukLiDsl4Tssq3rgpwg7Zi8656A7J2YIOyLnOuMgOuLpC4ifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50Ijoi67mE7KaI64uI7Iqk64qU7JeQIOuMgO2VtCDsnpDshLjtnogg7JWM66Ck7KSYLiJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiLruYTspojri4jsiqTripQg7LSI7KSR6rOgwrfrjIDtlZkg7Ja065SU7ISc64+EIOuwsOyatCDsoIHsnbQg7JeG64ukLiDqsozri6TqsIAg7J207KCc64qUIOyCrOuejOydhCDrp47snbQg7JOw7KeAIOyViuuKlCBBSSDquLDrsJggMeyduCDquLDsl4Ug7ZiV7YOc66GcIO2VtOyVvCDtlZzri6QuIn1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6IkFJ7JeQIOuMgO2VtCDslYzroKTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IkFJIOyLnOuMgCDqs7XrtoDrspUgNOuLqOqzhDog4pGgIEdlbmVyYXRpb24o7IOd7ISxKSDikaEgU3lzdGVtKOyLnOyKpO2FnCkg4pGiIEJ1aWxkKOq1rOy2lSkg4pGjIEF1dG9tYXRpb24o7J6Q64+Z7ZmUKS4g7ZWZ6rWQIOuLpOuLiOuptCDsgqzsnbTrk5wg7ZSE66Gc7KCd7Yq466GcLCDtmozsgqwg64uk64uI66m0IOu2gOyXheyymOufvCDqs7XrtoDtlZzri6QuIn1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6IuKRoCDqtIDroKjtlbTshJwg7ISk66qF7ZW07KSYLiJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiLikaAgR2VuZXJhdGlvbiDigJQg66+47IigwrfsnYzslYXCt+q4gOyTsOq4sCDrk7Eg7IOd7ISxIOuwqeyLneydtCDri6Qg67CU64CM7JeI64ukLiDri6jsiJztnogg66eM65Ok7KeAIOunkOqzoCwg7ZKA66Ck64qUIOusuOygnMK37ISc67mE7Iqk7J2YIOu5hOyaqeqzvCDsiJjsnbUoUk9JKeydhCDruYTspojri4jsiqQg66eI7J2465Oc66GcIOygle2Zle2eiCDqs4TsgrDtlaAg7KSEIOyVjOyVhOyVvCDtlZzri6QuIn1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6IuyDneyEseydtCjqsIApIOutlOyngCDslYzroKTspITrnpg/In0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IuyDneyEsSDrj4Tqtaw6IO2FjeyKpO2KuMK37L2U65Sp7J2AIENoYXRHUFTCt0NsYXVkZeqwgCDqsJXtlZjqs6AsIOydtOuvuOyngMK37JiB7IOBwrfsnYzslYXsnYAg67mE7JqpIO2BsCDrqqjrjbjsnbTrnbwg6rWs6riA7J20IOuPheygkOyggeydtOuLpCDigJQg7J2066+47KeAPeuCmOuFuOuwlOuCmOuCmCwg7JiB7IOBPVZlbyAzLjEsIOydjOyVhT1MeXJpYS4gQVBJIOqwgOqyqeydhCDsp4HsoJEg6rOE7IKw7ZW067SQ7JW8IO2VnOuLpC4ifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50Ijoi4pGh7JeQIOuMgO2VtCDrhKTqsIAg7JWE64qUIOqxuCDrp5DtlbTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IuKRoSBTeXN0ZW0g4oCUIOyCrOuejC3sgqzrnozsnbQg7JWE64uI6528IOyXkOydtOyghO2KuC3sl5DsnbTsoITtirgg7ZiR7JeFIOq1rOyhsOulvCDrsLDsm4zslbwg7ZWc64ukLiBuOG7Ct01ha2XCt0dvb2dsZeyymOufvCDrhbjrk5wo7Z2Q66aEKSDtmJXtg5zqsIAg7KeB6rSA7KCB7J206rOgLCDqtazquIAg64+E6rWs64qUIOustOujjOudvCDstpTsspztlZzri6QuIn1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6IuyLnOyKpO2FnOydhCjrpbwpIOygleumrO2VtOyEnCDshKTrqoXtlbTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IuyLnOyKpO2FnCDsmIjsi5w6IOq4sO2ajSDsl5DsnbTsoITtirjqsIAg7Yq466CM65OcKOudvOuptMK36rCV7JWE7KeAKeulvCDssL7snLzrqbQg7J2066+47KeAIOyXkOydtOyghO2KuOqwgCBcIuqwleyVhOyngOqwgCDrnbzrqbQg66i564qUIOq3uOumvFwi7J2EIOunjOuTpOqzoCDsnYzslYUg7JeQ7J207KCE7Yq46rCAIOyWtOyauOumrOuKlCBCR03snYQg7IOd7ISx7ZWc64ukLiA166qF7J20IO2VmOuNmCDsnbzsnYQg7JeQ7J207KCE7Yq4IO2YkeyXheycvOuhnC4ifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50Ijoi4pGi7JeQIOuMgO2VtCDsnpDshLjtnogg7JWM66Ck7KSYLiJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiLikaIgQnVpbGQg4oCUIOuwlOydtOu4jCDsvZTrlKnsnLzroZwg7Ju57IKs7J207Yq4wrfslbHsnYQg7KeB7KCRIOunjOuToOuLpC4gXCLrgrQg66eQ64yA66GcIOunjOuTpOyWtOyjvOuKlFwiIOyLnOuMgOuLpC4g7JWI7Yuw6re4656Y67mE7Yuw64qUIOq1rOq4gOydtCDsnbjsiJjtlZwg64+E6rWs66GcIOuCmOuFuOuwlOuCmOuCmCDrgrTsnqXCt+ustOujjOydtOqzoCDqsrDsoJzCt0RCIOyXsOqysOq5jOyngCDsnpgg64+8IOyeiOyWtCDstpTsspztlZzri6QuIn1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6IuKRo+yXkCDrjIDtlbQg7JWM66Ck7KSYLiJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiLikaMgQXV0b21hdGlvbiDigJQgMeyduCDquLDsl4XsnYQg7J6Q64+Z7ZmU7ZWc64ukLiDrqqntkZzripQg6rWs66mN6rCA6rKM6rCAIOyVhOuLiOudvCBcIu2YvOyekCDsmrTsmIHtlZjripQg7IK87ISx6riJIO2ajOyCrFwi64ukLiDslYjti7Dqt7jrnpjruYTti7Ag7Jik7ZSI66ek64uI7KCA66GcIOyXrOufrCDsl5DsnbTsoITtirjrpbwg66eM65Ok7Ja0IOuqheuguSDtlZwg67KI7JeQIOyKpOyKpOuhnCDtmJHsl4XtlZjqsowg7ZWc64ukLiJ9XX0KeyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiLsoJXri7Ug6rSA66Co7ZW07IScIOyEpOuqhe2VtOykmC4ifSx7InJvbGUiOiJhc3Npc3RhbnQiLCJjb250ZW50Ijoi7KCV64u1IOyXhuuKlCDsi5zrjIDsnZgg7ZW17IusID0g7J286rSA7ISxKGNvbnNpc3RlbmN5KeqzvCDsirXqtIAoaGFiaXQpLiDshLjsg4HsnYAg7ZmV66Wg7J206528IOyEseqztSDtmZXrpaDsnYAg7JW9IDElLiAxMDAw67KIIOyLnOuPhO2VtCAx67KIIOyEseqzte2VmOuKlCDqsowg7KeA6riIIOyEuOyDgeydtOuLpC4ifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50Ijoi7ISx6rO17ZWc7J20KOqwgCkg662U7KeAIOyVjOugpOykhOuemD8ifSx7InJvbGUiOiJhc3Npc3RhbnQiLCJjb250ZW50Ijoi7ISx6rO17ZWcIOyCrOuejOydvOyImOuhnSBcIuyatOydtOuLpFwi65286rOgIO2VnOuLpC4g7KCV64u17J20IOyXhuuLpOuKlCDqsbQg7IS47IOB7J20IO2ZleuloOyehOydhCDsnbjsoJXtlZjripQg6rKDLiDsnKDtipzruIwg7KGw7ZqM7IiY6rCAIOuqh+yLreunjH4xNTAw7J2EIOyYpOqwgOuKlCDqsoPrj4Qg6rCZ7J2AIOyCrOuejMK367mE7Iq37ZWcIO2AhOumrO2LsOyduOuNsCDtmZXrpaAg65WM66y47J2064ukLiJ9XX0KeyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiLqt7jrnpjshJzsl5Ag64yA7ZW0IOuEpOqwgCDslYTripQg6rG4IOunkO2VtOykmC4ifSx7InJvbGUiOiJhc3Npc3RhbnQiLCJjb250ZW50Ijoi6re4656Y7IScIOyhsO2ajOyImMK37IiY7J217JeQIOynkeywqe2VmOyngCDrp5Dqs6AgXCLsg53shLHihpLsi5zrj4TihpLsl4XroZzrk5xcIuudvOuKlCDsnpHsnYAg7ZaJ64+ZKGFjdGlvbinsnYQg66ek7J28IOuwmOuzte2VtCDsirXqtIDCt+ydvOq0gOyEseydhCDrp4zrk6Dri6QuIOyekeydgCDtlonrj5koYWN0aW9uKeydtCDsg4Htg5woc3RhdGUp66W8IOunjOuTpOqzoCwg7IyT7J2066m0IOyatOydtCDrqqjsnbjri6QuIn1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6IuuwseydhCjrpbwpIOygleumrO2VtOyEnCDshKTrqoXtlbTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IuuwsSDrsogg7LKcIOuyiCDsi5zrj4TtlbTshJwg7JWIIOuQnCDsgqzrnozsnYQg67O4IOyggeydtCDsl4bri6QuIOuLpOunjCDrsLEg67KIIOyynCDrsogg7ZWY64qUIOyCrOuejOydhCDrp4zrgpjquLDqsIAg7Ja066C164ukLiDqvrjspIDtlago7J286rSA7ISxKeydtCDsp4Tsp5wg7LCo67OE7KCQ7J2064ukLiJ9XX0KeyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiLqtZDsnKHsnYDsl5Ag64yA7ZW0IOyekOyEuO2eiCDslYzroKTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6Iuq1kOycoeydgCDrrLTrhIjsoYzsp4Drp4wg7IiY64qlIOyDne2DnOqzhCjtlZzqta3Ct+uvuOq1rSBTQVQp6rCAIOqxsOuMgO2VtCDtlZwg67KI7JeQIOyViCDrsJTrgIzqs6Ag7Jik656YIOqxuOumsOuLpC4g7IS47IOB7J20IOyViCDrsJTrgIzslrTrj4Qg7IOd7KG06rO8IOyngeqysOuQmOuLiCDsmrDrpqzqsIAg66i87KCAIOuwlOuAjOyWtOyVvCDtlZzri6QuIn1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6IkFJ7JeQIOuMgO2VtCDslYzroKTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IkFJIOqzteu2gOuKlCDri6jsiJwg7ZWZ7Iq17J20IOyVhOuLiOudvCDtla3sg4Eg67mE7JqpwrfsiJjsnbXsnYQg6rOE7IKw7ZWY66mwIOyCrOyXhe2ZlO2VmOuKlCDqsoMuIOustOyXh+ydhCDslrTrlqQgQUnroZwg7Ja866eI7JeQIO2SgOyngOulvCBcIjElw5cxMDAw67KIXCIg7KCE7KCc66GcIOqzhOyCsMK36rCc7ISgwrfsirXqtIDtmZTtlZjripQg6rKMIOyngOq4iCDtlYTsmpTtlZwg6rWQ7Jyh7J2064ukLiJ9XX0KeyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiIxMH43MOuMgCDqtIDroKjtlbTshJwg7ISk66qF7ZW07KSYLiJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiIxMH43MOuMgCDrqqjrkZAg7JWM7JWE7JW8IO2VnOuLpC4gMTB+MzDrjIDripQgQUnrpbwg67mo66asIOuwsOyasOuLiCDruYTspojri4jsiqTrpbwg7J217Z6I6rOgLCA1MH43MOuMgOuKlCDtko3rtoDtlZwg6rK97ZeYKOu2iO2OuMK367aI66eMIO2VtOqysCnsnYQgQUnroZwg7LC97JeF7JeQIO2ZnOyaqe2VmOudvC4g7IKs656MIOuMgOyLoCBBSSDsl5DsnbTsoITtirjrpbwg6rOg7Jqp7ZW0IDHsnbgg6riw7JeF7J2EIOunjOuTnOuKlCDsi5zrjIDri6QuIn1dfQ=="
open("brain.jsonl", "w").write(base64.b64decode(_B64).decode("utf-8"))
ds = load_dataset("json", data_files="brain.jsonl", split="train")
# qwen Instruct 모델은 내장 chat_template 사용(별도 변환 불필요)
def fmt(ex):
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix("<bos>") for c in ex["conversations"]]
    return {"text": texts}
ds = ds.map(fmt, batched=True)
print("데이터 개수:", len(ds)); print(ds[0]["text"][:400])


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1, gradient_accumulation_steps = 4,
        warmup_steps = 5, max_steps = 52, learning_rate = 0.0002,
        logging_steps = 1, optim = "adamw_8bit", weight_decay = 0.001,
        lr_scheduler_type = "linear", seed = 3407, report_to = "none",
    ),
)


In [ ]:
# 🎭 응답(assistant)만 학습 — 질문 패턴은 마스킹(효율↑·품질↑)
# ⚠️ 마커는 계열마다 다름 → 실제 텍스트에서 자동 감지 (gemma·llama·qwen 모두 지원)
from unsloth.chat_templates import train_on_responses_only
_t = ds[0]["text"]
if "<start_of_turn>user" in _t: _im, _rm = "<start_of_turn>user\n", "<start_of_turn>model\n"
elif "<|start_header_id|>" in _t: _im, _rm = "<|start_header_id|>user<|end_header_id|>\n\n", "<|start_header_id|>assistant<|end_header_id|>\n\n"
elif "<|im_start|>" in _t: _im, _rm = "<|im_start|>user\n", "<|im_start|>assistant\n"
elif "<|turn>user" in _t: _im, _rm = "<|turn>user\n", "<|turn>model\n"
else: _im, _rm = None, None
if _rm:
    trainer = train_on_responses_only(trainer, instruction_part=_im, response_part=_rm)
    print(f"✅ 마스킹 마커 자동감지: {_rm.strip()} — 응답만 학습")
else:
    print("ℹ️ 마커 자동감지 실패 → 전체 텍스트로 학습(문제 없음)")


In [ ]:
trainer_stats = trainer.train()
print("🎉 학습 완료! 최종 loss:", round(trainer_stats.training_loss, 4))
print("💡 loss 0.2~0.4면 sweet spot. 너무 낮으면(<0.1) 과적합 — max_steps 줄이세요.")


## 🧪 학습된 모델 테스트 (업로드 전에 확인!)
내가 가르친 지식을 직접 물어보세요. 답에 그 내용이 나오면 학습 성공이에요. 질문은 자유롭게 바꿔도 됩니다.


In [ ]:
from unsloth import FastModel
FastModel.for_inference(model)
def chat(prompt, max_tokens=220):
    try:
        msg = [{"role":"user","content":[{"type":"text","text":prompt}]}]
        inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    except Exception:
        msg = [{"role":"user","content":prompt}]
        inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    inp = inp.to(model.device)
    if inp["input_ids"][0,0].item() == tokenizer.bos_token_id:
        inp["input_ids"] = inp["input_ids"][:,1:]; inp["attention_mask"] = inp["attention_mask"][:,1:]
    out = model.generate(**inp, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    ans = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\u2753 {prompt}\n\U0001F4AC {ans}\n" + "\u2500"*58)

# 👇 내가 가르친 지식에 대해 물어보세요 (자유롭게 수정)
chat("내 사업/지식에 대해 아는 걸 알려줘")
chat("너는 무엇을 도와줄 수 있어?")


## 💾 저장 → HuggingFace
**safetensors(AI 진화·합성용) + GGUF(앱 실행용)** 둘 다 올라가요. (맨 앞에서 로그인했으니 바로 됩니다)


In [ ]:
# 메모리 정리(OOM 방지) — 학습기 메모리 해제 후 변환
import gc, torch
try:
    del trainer
except Exception:
    pass
gc.collect(); torch.cuda.empty_cache()
# 📤 저장 위치 = "내" HF 계정 (위에서 로그인한 본인 계정으로 자동 — 노트북이 공유돼도 안전)
from huggingface_hub import HfApi
NAME = "mybrain-lee"
OUTPUT = f'{HfApi().whoami()["name"]}/{NAME}'
print("📤 내 계정에 저장:", OUTPUT)
# ① 합성용 safetensors (AI 진화소에서 다시 합칠 수 있어요 — 이게 없으면 합성 불가!)
try:
    model.push_to_hub_merged(OUTPUT, tokenizer, save_method="merged_16bit", token=True)
    print("✅ safetensors 업로드 — AI 진화소에서 합치기 가능")
except Exception as e:
    print("⚠️ 병합 업로드 실패 → 어댑터(LoRA)로 폴백:", e)
    model.push_to_hub(OUTPUT, token=True); tokenizer.push_to_hub(OUTPUT, token=True)
# ② 앱 실행용 GGUF
model.push_to_hub_gguf(OUTPUT, tokenizer, quantization_method="q4_k_m", token=True)
print(f"✅ 완료! safetensors(합성용)+GGUF(실행용) 둘 다 → Connect AI 앱 🤖 내 AI 에서 {OUTPUT} 받기")
